# Capítulo 3 — Probabilidade e Estatística para Finanças

Este capítulo trata incerteza como objeto matemático e financeiro. A sequência conceitual é guiada pelo Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander, sem reproduzir o texto da obra.

**Objetivos:**

- descrever amostras de retornos por momentos, quantis e distribuições;
- entender quando a Normal é uma aproximação ruim para dados financeiros;
- estimar parâmetros por máxima verossimilhança e testar hipóteses;
- modelar dependência multivariada e eventos extremos;
- simular random walk, processos estacionários, mean reversion, GBM e saltos.

A prioridade é interpretar cada modelo antes de chamar uma função de `scipy.stats`. Distribuições são aproximações úteis, não verdades sobre o mercado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from scipy.stats import gaussian_kde, genextreme, genpareto

from quantfinance.simulation import (
    simulate_gbm,
    simulate_jump_diffusion,
    simulate_ou,
    simulate_random_walk,
)
from quantfinance.statistics import (
    descriptive_statistics,
    fit_normal_mle,
    fit_student_t,
    historical_quantile,
    normal_log_likelihood,
    return_summary,
)

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 3.1 — Variáveis aleatórias, amostras, PDF, CDF e momentos

Uma variável aleatória associa resultados numéricos a eventos incertos. Uma distribuição descreve como sua probabilidade se espalha. Para uma variável contínua, a PDF $f(x)$ é uma densidade e a CDF acumula probabilidade:

$$F(x)=P(X\le x)=\int_{-\infty}^{x} f(u)\,du.$$

Uma amostra é uma realização finita dessa variável. A média resume localização; a variância mede dispersão; skewness mede assimetria; excesso de kurtosis mede peso relativo das caudas:

$$\bar x=\frac{1}{n}\sum_i x_i, \qquad s^2=\frac{1}{n-1}\sum_i(x_i-\bar x)^2.$$

Quantis e percentis são mais robustos para risco de cauda do que a média. O percentil 5% é o quantil $q_{0.05}$.

**Aplicação financeira:** retornos diários são amostras de uma variável aleatória; o quantil inferior pode representar uma perda histórica extrema.

**Exercício:** compare média, mediana, desvio padrão e quantis de uma amostra Normal com uma amostra Student-t.

In [ ]:
rng = np.random.default_rng(42)
heavy_tailed_returns = 0.01 * rng.standard_t(df=5, size=5_000)
summary = descriptive_statistics(heavy_tailed_returns)
summary_table = pd.Series(summary)
display(summary_table.to_frame("valor"))
print("Percentil 1%:", historical_quantile(heavy_tailed_returns, 0.01))
print("Percentil 99%:", historical_quantile(heavy_tailed_returns, 0.99))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heavy_tailed_returns, bins=60, density=True, alpha=0.75)
axes[0].set_title("Histograma de retornos Student-t")
axes[0].set_xlabel("Retorno")
axes[1].plot(np.sort(heavy_tailed_returns), np.linspace(0, 1, len(heavy_tailed_returns)))
axes[1].set_title("CDF empírica")
axes[1].set_xlabel("Retorno")
axes[1].set_ylabel("F(x)")
plt.tight_layout()
plt.show()

## 3.2 — Famílias de distribuições e aplicações financeiras

| Distribuição | Parâmetros | Média e variância | Formato e uso financeiro |
|---|---|---|---|
| Binomial | $n,p$ | $np$, $np(1-p)$ | discreta; número de sucessos em cenários de crédito ou default |
| Poisson | $\lambda$ | $\lambda$, $\lambda$ | discreta; contagem de eventos raros por intervalo |
| Exponencial | $\lambda$ | $1/\lambda$, $1/\lambda^2$ | espera entre eventos; tempo até ocorrência |
| Uniforme | $a,b$ | $(a+b)/2$, $(b-a)^2/12$ | incerteza limitada; cenários simples, não caudas reais |
| Normal | $\mu,\sigma$ | $\mu$, $\sigma^2$ | simétrica; aproximação para choques pequenos |
| Lognormal | $\mu,\sigma$ no log | $e^{\mu+\sigma^2/2}$, fórmula lognormal | positiva; preços em modelos multiplicativos |
| Mistura Normal | pesos e parâmetros por regime | média/variância combinadas | regimes de mercado, podendo gerar assimetria e caudas |
| Student-t | $\nu,\mu,s$ | $\mu$, $\nu s^2/(\nu-2)$ se $\nu>2$ | simétrica com caudas pesadas; retornos e risco extremo |
| GEV/GPD | forma, localização, escala | dependem da forma | máximos ou excessos; EVT, com forte dependência do threshold |

A Normal falha quando há assimetria, curtose elevada, volatilidade variável, dependência temporal ou eventos extremos. A Student-t melhora caudas simétricas, mas não resolve assimetria nem clusters de volatilidade. Distribuições estáveis podem representar caudas ainda mais pesadas, porém o ajuste é sensível e a variância pode não existir; usaremos `levy_stable` apenas como demonstração opcional.

**Exercício:** escolha uma distribuição para contar eventos, uma para tempos de espera e uma para retornos; justifique os parâmetros e as limitações.

In [ ]:
x = np.linspace(-0.06, 0.06, 500)

normal_pdf = stats.norm.pdf(x, loc=0, scale=0.01)
t_pdf = stats.t.pdf(x / 0.01, df=5) / 0.01
lognormal_x = np.linspace(0.001, 0.08, 500)
lognormal_pdf = stats.lognorm.pdf(lognormal_x, s=0.25, scale=np.exp(0.0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, normal_pdf, label="Normal")
axes[0].plot(x, t_pdf, label="Student-t, df=5")
axes[0].set_title("PDF: Normal versus Student-t")
axes[0].legend()
axes[1].plot(lognormal_x, lognormal_pdf, label="Lognormal")
axes[1].plot(np.linspace(0, 0.08, 500), stats.uniform.pdf(np.linspace(0, 0.08, 500), 0, 0.08), label="Uniforme")
axes[1].set_title("PDFs com suporte positivo/limitado")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Binomial P(X=3):", stats.binom.pmf(3, n=10, p=0.4))
print("Poisson P(X=2):", stats.poisson.pmf(2, mu=1.5))
print("Exponencial CDF em t=2:", stats.expon.cdf(2, scale=1 / 0.5))
print("Normal CDF em zero:", stats.norm.cdf(0))

## 3.2.1 — Distribuições estáveis: demonstração opcional

A família estável pode acomodar caudas muito pesadas e assimetria, mas nem sempre possui média ou variância finitas. Por isso, não deve ser usada automaticamente para calcular volatilidade ou Sharpe. A implementação `levy_stable` do SciPy é adequada aqui apenas como demonstração exploratória; o ajuste pode ser lento e numericamente sensível.

**Exercício:** simule uma estável simétrica e compare seus quantis extremos com os da Normal, sem interpretar a variância amostral como parâmetro estrutural.

In [ ]:
stable_sample = stats.levy_stable.rvs(
    alpha=1.8, beta=0.0, loc=0.0, scale=0.01, size=5_000, random_state=rng
)
print("Quantis estáveis 1% e 99%:", np.quantile(stable_sample, [0.01, 0.99]))
print("Amostra estável gerada; média/variância teóricas não são assumidas.")

## 3.3 — Comparação empírica: Normal versus Student-t

Vamos simular retornos com as mesmas escala e localização. A comparação deve olhar além do histograma: skewness, excesso de kurtosis, QQ plot, likelihood e quantis extremos.

A Normal tem curtose de referência igual a zero em excesso. A Student-t tem caudas mais pesadas; sua variância só existe para $\nu>2$ e sua curtose só é finita para $\nu>4$.

**Interpretação financeira:** um modelo Normal pode subestimar a frequência de perdas e ganhos extremos quando os dados têm caudas pesadas.

**Exercício:** repita a comparação com $\nu=3$, lembrando que a variância teórica continua finita, mas a curtose teórica não.

In [ ]:
normal_sample = rng.normal(0.0, 0.01, 5_000)
t_sample = 0.01 * rng.standard_t(df=5, size=5_000)
normal_fit = fit_normal_mle(normal_sample)
t_fit = fit_student_t(t_sample)

normal_summary = descriptive_statistics(normal_sample)
t_summary = descriptive_statistics(t_sample)
comparison = pd.DataFrame({"Normal": normal_summary, "Student-t": t_summary})
display(comparison.loc[["skewness", "excess_kurtosis", "quantile_01", "quantile_99"]])
print("Normal log-likelihood:", normal_fit["log_likelihood"])
print("Student-t log-likelihood:", t_fit["log_likelihood"])
print("Student-t graus de liberdade ajustados:", t_fit["degrees_of_freedom"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stats.probplot(normal_sample, dist="norm", plot=axes[0])
axes[0].set_title("QQ plot: Normal")
stats.probplot(t_sample, dist="norm", plot=axes[1])
axes[1].set_title("QQ plot: Student-t contra Normal")
plt.tight_layout()
plt.show()

## 3.4 — EVT, GEV, GPD e KDE

A Teoria de Valores Extremos separa o comportamento de máximos (GEV) do comportamento dos excessos acima de um threshold (GPD). Para um excesso $Y=X-u$ condicionado a $X>u$:

$$P(Y\le y\mid X>u)\approx G_{\xi,\beta}(y).$$

O threshold deve ser alto o suficiente para a aproximação assintótica, mas não tão alto que deixe poucas observações. O ajuste GPD é sensível ao threshold, à dependência temporal e à não estacionariedade. EVT não prevê crises por si só e não elimina risco de extrapolação.

A KDE estima uma densidade suave sem impor uma família paramétrica:

$$\hat f_h(x)=\frac{1}{nh}\sum_{i=1}^nK\left(\frac{x-x_i}{h}\right).$$

**Interpretação financeira:** KDE ajuda a visualizar a forma da distribuição; GPD ajuda a estudar perdas além de um limiar, mas requer diagnóstico de estabilidade.

**Exercício:** repita o ajuste GPD com thresholds de 90%, 95% e 99% e compare forma, escala e número de excessos.

In [ ]:
mu = np.array([0.001, 0.0005])
V = np.array([[0.0004, 0.00018],
              [0.00018, 0.0009]])

normal_multivariate = rng.multivariate_normal(mu, V, size=5_000)
correlation_matrix = V / np.outer(np.sqrt(np.diag(V)), np.sqrt(np.diag(V)))
normal_df = pd.DataFrame(normal_multivariate, columns=["R1", "R2"])

degrees_of_freedom = 5
standard_normal = rng.multivariate_normal([0.0, 0.0], correlation_matrix, size=5_000)
chi_square = rng.chisquare(degrees_of_freedom, 5_000)
t_multivariate = standard_normal / np.sqrt(chi_square[:, None] / degrees_of_freedom)
t_df = pd.DataFrame(t_multivariate, columns=["R1", "R2"])

print("Covariância Normal multivariada:")
display(normal_df.cov())
print("Correlação Normal multivariada:")
display(normal_df.corr())
print("Correlação Student-t multivariada:")
display(t_df.corr())

## 3.6 — Intervalos de confiança, CLT e testes

O Teorema Central do Limite diz que, sob condições usuais, a média amostral tende a uma Normal quando o tamanho da amostra cresce, mesmo que os dados individuais não sejam Normais. Isso justifica intervalos aproximados, mas a convergência pode ser lenta para caudas pesadas.

Para uma média com desvio padrão estimado $s$:

$$\bar X\pm t_{n-1,1-\alpha/2}\frac{s}{\sqrt n}.$$

Um teste de hipótese compara uma estatística observada com a distribuição esperada sob $H_0$. O p-valor não mede a probabilidade de $H_0$ ser verdadeira.

O teste t é paramétrico. O teste de Mann-Whitney compara posições entre dois grupos sem exigir Normalidade, mas também depende de hipóteses próprias e não é uma solução universal para dependência temporal.

**Exercício:** repita o intervalo de confiança com amostras Student-t de tamanhos 30, 100 e 1.000 e observe a aproximação à Normal.

In [ ]:
sample_size = 100
sample_for_inference = rng.standard_t(df=5, size=sample_size) * 0.02 + 0.001
sample_mean = sample_for_inference.mean()
standard_error = stats.sem(sample_for_inference)
confidence_interval = stats.t.interval(
    0.95, df=sample_size - 1, loc=sample_mean, scale=standard_error
)

print("Média amostral:", sample_mean)
print("IC 95% para a média:", confidence_interval)

null_t, null_pvalue = stats.ttest_1samp(sample_for_inference, popmean=0.0)
print("Teste t H0: média = 0 | t =", null_t, "p =", null_pvalue)

first_group = rng.normal(0.001, 0.01, 100)
second_group = rng.standard_t(df=5, size=100) * 0.01 + 0.001
mann_whitney = stats.mannwhitneyu(first_group, second_group, alternative="two-sided")
print("Mann-Whitney U:", mann_whitney.statistic, "p =", mann_whitney.pvalue)

sample_means = [
    rng.standard_t(df=5, size=500).mean()
    for _ in range(2_000)
]
plt.figure(figsize=(8, 4))
plt.hist(sample_means, bins=50, density=True)
plt.title("CLT: distribuição das médias amostrais")
plt.xlabel("Média amostral")
plt.show()

## 3.7 — Máxima verossimilhança

A verossimilhança mede quão compatíveis são os dados com parâmetros propostos. Trabalhamos com log-likelihood para transformar produtos em somas:

$$\ell(\theta)=\sum_{i=1}^n\log f(x_i\mid\theta).$$

Para a Normal, o MLE da média é a média amostral e o MLE da escala usa divisor $n$, não $n-1$. Para Student-t, os parâmetros são ajustados numericamente porque os graus de liberdade também são desconhecidos.

**Interpretação financeira:** comparar likelihoods ajuda a avaliar se uma Student-t descreve melhor caudas pesadas do que uma Normal, mas não substitui validação fora da amostra.

**Exercício:** ajuste Normal e Student-t a outra amostra e compare quantis de 1% e 99%, não apenas o likelihood.

In [ ]:
data = 0.01 * rng.standard_t(df=5, size=1_000) + 0.002
normal_fit = fit_normal_mle(data)
student_fit = fit_student_t(data)

fit_table = pd.DataFrame({"Normal": normal_fit, "Student-t": student_fit})
display(fit_table)

normal_01 = stats.norm.ppf(0.01, loc=normal_fit["mean"], scale=normal_fit["standard_deviation"])
student_01 = stats.t.ppf(
    0.01,
    student_fit["degrees_of_freedom"],
    loc=student_fit["location"],
    scale=student_fit["scale"],
)
print("Quantil 1% Normal ajustada:", normal_01)
print("Quantil 1% Student-t ajustada:", student_01)
print("Likelihood Normal:", normal_fit["log_likelihood"])
print("Likelihood Student-t:", student_fit["log_likelihood"])
print("Likelihood Normal por função:", normal_log_likelihood(
    data, normal_fit["mean"], normal_fit["standard_deviation"]
))

## 3.8 — Random walk, estacionariedade, mean reversion, GBM e saltos

Um random walk aditivo acumula choques e não é estacionário em nível. Um processo estacionário tem distribuição que não muda no tempo; o processo Ornstein-Uhlenbeck (OU) é mean-reverting:

$$dX_t=\kappa(\mu-X_t)dt+\sigma dW_t.$$

O GBM modela preços positivos com crescimento multiplicativo:

$$dS_t=\mu S_tdt+\sigma S_tdW_t.$$

Um processo de jump-diffusion acrescenta saltos gerados por um processo de Poisson:

$$d\log S_t=\left(\mu-\frac{1}{2}\sigma^2\right)dt+\sigma dW_t+JdN_t.$$

**Interpretação financeira:** OU é adequado para spreads ou taxas com reversão; GBM é um modelo de referência para preços positivos; saltos representam notícias e movimentos discretos. Nenhum deles captura sozinho todos os padrões de mercado.

**Exercício:** compare a dispersão final de um random walk e de um OU; depois aumente a intensidade de saltos e observe a trajetória de preços.

In [ ]:
random_walk = simulate_random_walk(
    n_steps=1_000, drift=0.0001, volatility=0.01, start=0.0, seed=42
)
ou_path = simulate_ou(
    mean_reversion=2.0,
    long_run_mean=0.0,
    volatility=0.30,
    horizon=1.0,
    steps=1_000,
    start=0.5,
    seed=42,
)
gbm_path = simulate_gbm(
    spot=100.0,
    drift=0.08,
    volatility=0.20,
    horizon=1.0,
    steps=252,
    seed=42,
)

plt.figure(figsize=(10, 4))
plt.plot(random_walk, label="Random walk")
plt.plot(ou_path, label="OU mean-reverting")
plt.legend()
plt.title("Random walk versus processo estacionário OU")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(gbm_path)
plt.title("Trajetória de GBM")
plt.xlabel("Passo")
plt.ylabel("Preço")
plt.show()

print("Desvio padrão da primeira metade do OU:", np.std(ou_path[:500]))
print("Desvio padrão da segunda metade do OU:", np.std(ou_path[500:]))

In [ ]:
jump_diffusion_path = simulate_jump_diffusion(
    spot=100.0,
    drift=0.08,
    volatility=0.20,
    jump_intensity=2.0,
    jump_mean=-0.05,
    jump_volatility=0.10,
    horizon=1.0,
    steps=252,
    seed=42,
)

plt.figure(figsize=(10, 4))
plt.plot(jump_diffusion_path)
plt.title("GBM com saltos de um processo de Poisson")
plt.xlabel("Passo")
plt.ylabel("Preço")
plt.show()

print("Preço inicial:", jump_diffusion_path[0])
print("Preço final:", jump_diffusion_path[-1])
print("A trajetória permanece positiva:", np.all(jump_diffusion_path > 0))

## Exercícios integradores

1. Compare a média e os quantis de uma amostra Normal e de uma Student-t.
2. Explique por que skewness e excesso de kurtosis ajudam a diagnosticar falhas da Normal.
3. Gere PMFs de Binomial e Poisson e relacione seus parâmetros a eventos financeiros.
4. Compare uma KDE com uma densidade Normal ajustada.
5. Repita o ajuste GPD em diferentes thresholds e discuta estabilidade.
6. Construa um intervalo de confiança para a média e interprete seu significado corretamente.
7. Compare um teste t com Mann-Whitney e descreva as hipóteses de cada um.
8. Compare likelihood e quantis extremos de Normal e Student-t ajustadas.
9. Explique por que um random walk não é estacionário e por que OU é mean-reverting.
10. Aumente a intensidade de saltos e avalie a diferença entre GBM e jump-diffusion.